In [1]:
%load_ext autoreload

In [ ]:
%autoreload 2
import os
from pathlib import Path

from darpinstances.instance import load_instance_config
from darpinstances.instance_generation.generate_config import generate_config

PATH = Path.cwd()
INSTANCE_PATH = PATH.parents[2] / "Instances"
# RESULTS_PATH = PATH.parents[2] / "Results"
INSTANCE_PATH_NEW = PATH.parents[2] / "generated" / "Instances"
RESULTS_PATH = PATH.parents[2] / "generated" / "Results"

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/home/dominika/Desktop/deathOFbachelor/Ridesharing_DARP_instances/python/darpinstances/instance.py:23: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
def create_custom_results_config(instance_path, method, outdir):
    config = {}
    config['instance'] = instance_path
    config['method'] = method
    config['outdir'] = outdir
    return config

In [6]:
# cities = ['Porto']
cities = ['Porto', 'Sydney', 'DC', 'Manhattan']
# cities = ['Porto', 'Sydney', 'NYC', 'DC', 'Chicago', 'Manhattan']
starts_str = {'18-00': ['05_min', '15_min', '30_min', '2_h']}
starts = {18: [5, 15, 30, 120]}
# start_durations = {7: [16*60], 18: [180, 1, 30, 15, 5]}
# delays_str = ['03_min', '05_min', '10_min']
delays_str = ['03_min', '05_min', '10_min', '15_min']
# delays = [3, 5, 10]
delays = [3, 5, 10, 15]
methods = ['ih', 'vga', 'halns', 'vga_chaining']
capacities = [4, 6, 10]

files_copy = ['requests.csv', 'trips.di', 'vehicles.csv']
dir_copy = "shapefiles"

rci = True
instance_setup = True

In [ ]:
from itertools import product
import shutil
ic = 0
rc = 0
for start, durations in starts.items():
    start_str = f"{start:02d}-00"

    for (i, duration), (j, delay), city, capacity in product(
        enumerate(durations), enumerate(delays), cities, capacities
    ):
        
        # setup new instance config
        instance_dir_new = INSTANCE_PATH_NEW / f'{city}/instances/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}'
        instance_path = instance_dir_new / "config.yaml"

        # load old config
        instance_dir_old = INSTANCE_PATH / f'{city}/instances/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[0]}'
        instance_config_path = instance_dir_old / 'config.yaml'
        if not instance_config_path.exists() and starts_str[start_str][i] == '2_h':
            instance_dir_old = INSTANCE_PATH / f'{city}/instances/start_{start_str}/duration_02_h/max_delay_{delays_str[0]}'
            instance_config_path = instance_dir_old / 'config.yaml'
            
        instance_config = load_instance_config(instance_config_path)

        # modify config values
        instance_config['vehicles']['vehicle_capacity'] = capacity
        instance_config['demand']['filepath'] = './requests.csv'
        instance_config['max_prolongation'] = delay*60
        instance_config['area_dir'] = "../../../../../"
        instance_config['vehicles'].pop('vehicle_count', None)
        instance_config['map'].pop('path', None)
        instance_config.pop('instance_dir', None)
        
        if instance_setup:
            os.makedirs(instance_dir_new, exist_ok=True)

            # generate new config
            generate_config(instance_config, instance_path)

            # copy files
            for f in files_copy:
                src = instance_dir_old / f
                dst = instance_dir_new / f
                if src.exists():
                    shutil.copy(src, dst)
            
            # copy directory
            src = instance_dir_old / dir_copy    # generate new config
            generate_config(instance_config, instance_path)

            # copy files
            for f in files_copy:
                src = instance_dir_old / f
                dst = instance_dir_new / f
                if src.exists():
                    shutil.copy(src, dst)
            
            # copy directory
            src = instance_dir_old / dir_copy
            dst = instance_dir_new / dir_copy
            if src.exists():
                shutil.copytr
            dst = instance_dir_new / dir_copy
            if src.exists():
                shutil.copytree(src, dst, dirs_exist_ok=True)
            
            ic += 1

        # setup new results configs
        for method in methods:

            results_dir = RESULTS_PATH / f'{city}/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}/{method}'
            os.makedirs(results_dir, exist_ok=True)
            results_path = results_dir / "config.yaml"

            if rci:
                instance_path =  f'/Instances/{city}/instances/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}/config.yaml'
                results_dir = f'/Results/{city}/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}/{method}'
            
            results_config = create_custom_results_config(str(instance_path), method, str(results_dir))

            results_dir = RESULTS_PATH / f'{city}/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}/{method}'
            
            generate_config(results_config, results_path)
            rc += 1

  
print(f'{ic} instance configs created')
print(f'{rc} instance configs created')

09:46:29 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_03_min/config.yaml
09:46:29 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/Porto/start_18-00/duration_05_min/max_delay_03_min/capacity_4/ih/config.yaml
09:46:29 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/Porto/start_18-00/duration_05_min/max_delay_03_min/capacity_4/vga/config.yaml
09:46:29 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/Porto/start_18-00/duration_05_min/max_delay_03_min/capacity_4/halns/config.yaml
09:46:29 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/Porto/start_18-00/duration_05_min/max_delay_03_min/capacity_4/vga_chaining/config.yaml
09:46:29 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_03_min/config.yaml


0 instance configs created
768 instance configs created


In [8]:
# replace demand filepaths in instance configs - trips.di for requests.csv
from itertools import product
ic = 0
for start, durations in starts.items():
    start_str = f"{start:02d}-00"

    for (i, duration), (j, delay), city, capacity in product(
        enumerate(durations), enumerate(delays), cities, capacities
    ):
        
        # setup new instance config
        instance_dir_new = INSTANCE_PATH_NEW / f'{city}/instances/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}'
        instance_path = instance_dir_new / "config.yaml"

        # load old config
        instance_dir_old = INSTANCE_PATH / f'{city}/instances/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[0]}'
        instance_config_path = instance_dir_old / 'config.yaml'
        if not instance_config_path.exists() and starts_str[start_str][i] == '2_h':
            instance_dir_old = INSTANCE_PATH / f'{city}/instances/start_{start_str}/duration_02_h/max_delay_{delays_str[0]}'
            instance_config_path = instance_dir_old / 'config.yaml'
            
        instance_config = load_instance_config(instance_config_path)

        # modify config values
        instance_config['vehicles']['vehicle_capacity'] = capacity
        instance_config['demand']['filepath'] = './requests.csv'
        instance_config['max_prolongation'] = delay*60
        instance_config['area_dir'] = "../../../../../"
        instance_config['vehicles'].pop('vehicle_count', None)
        instance_config['map'].pop('path', None)
        instance_config.pop('instance_dir', None)
        
        if instance_setup:
            os.makedirs(instance_dir_new, exist_ok=True)

            # generate new config
            generate_config(instance_config, instance_path)
            
            # copy directory
            src = instance_dir_old / dir_copy    # generate new config
            generate_config(instance_config, instance_path)
            ic += 1

print(f'{ic} instance configs created')

11:31:54 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_03_min/config.yaml
11:31:54 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_4/config.yaml
11:31:54 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_4/config.yaml
11:31:54 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_03_min/config.yaml


11:31:54 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_6/config.yaml
11:31:54 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_6/config.yaml
11:31:54 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_03_min/config.yaml
11:31:54 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_10/config.yaml
11:31:54 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_10/config.yaml
11:31:54 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_18-00/duration_05_min/max_

192 instance configs created
